In [12]:
import os
import json
import pandas as pd
import re

def analyze_sweeps(root_dir):
    results = []
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            # 排除 checkpoint 文件，只处理 jsonl
            if file.endswith(".jsonl") and "-checkpoint" not in file:
                file_path = os.path.join(root, file)
                
                # 提取任务名称 (假设在路径的第三层)
                path_parts = root.split(os.sep)
                task = path_parts[2] if len(path_parts) > 2 else "Unknown"
                
                # 正则匹配新格式: results_layer_12_alpha_0.5
                layer_match = re.search(r'layer_(\d+)', file)
                alpha_match = re.search(r'alpha_([\d\.]+)', file)
                
                layer = layer_match.group(1) if layer_match else "N/A"
                alpha = alpha_match.group(1) if alpha_match else "N/A"
                
                try:
                    correct_count = 0
                    total_count = 0
                    with open(file_path, 'r', encoding='utf-8') as f:
                        for line in f:
                            data = json.loads(line)
                            total_count += 1
                            if data.get("is_correct") is True:
                                correct_count += 1
                    
                    accuracy = (correct_count / total_count) * 100 if total_count > 0 else 0
                    
                    results.append({
                        "Task": task,
                        "Layer": layer,
                        "Alpha": alpha,
                        "Accuracy": accuracy,
                        "File": file
                    })
                except Exception as e:
                    print(f"Error processing {file_path}: {e}")

    df = pd.DataFrame(results)
    if df.empty:
        return None

    # --- 核心修改：将字符串转为数值，以便进行逻辑排序 ---
    df['Layer'] = pd.to_numeric(df['Layer'], errors='coerce')
    df['Alpha'] = pd.to_numeric(df['Alpha'], errors='coerce')

    # 按 Task 分组，然后按 Layer 和 Alpha 升序排列，方便对比
    df = df.sort_values(by=["Task", "Layer", "Alpha"], ascending=[True, True, True])
    
    return df

# 执行分析
dataset = "LogicalDeduction"  # 也可以换成 "FOLIO" 或 "ProofWriter"(LogicalDeduction FOLIO ProntoQA AR-LSAT ProofWriter)
root_path = "./static/Qwen2.5-3B-Instruct/LogicalDeduction" 
all_results_df = analyze_sweeps(root_path)

if all_results_df is not None:
    # --- 修改部分：按 Accuracy 降序排列 ---
    # ascending=False 表示降序
    all_results_df = all_results_df.sort_values(by='Accuracy', ascending=False)
    # ------------------------------------
    print(f"### 找到共 {len(all_results_df)} 条实验记录 ###")
    
    # 使用 pd.option_context 确保打印所有行，不会因为太长被折叠 (Optional)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
        print(all_results_df.to_string(index=False)) # 不打印索引号，更整洁
        
    # 如果你想顺便存个 CSV 方便在 Excel 看：
    # all_results_df.to_csv("all_experiment_results.csv", index=False)
else:
    print("未找到有效的结果文件。")

### 找到共 68 条实验记录 ###
               Task  Layer  Alpha  Accuracy                                      File
Qwen2.5-3B-Instruct   16.0    NaN 55.000000          results_layer_16_alpha_1.5.jsonl
Qwen2.5-3B-Instruct    NaN    NaN 53.666667        results_32baseline_alpha_0.0.jsonl
Qwen2.5-3B-Instruct    6.0    NaN 53.333333  instance_results_layer_6_alpha_1.0.jsonl
Qwen2.5-3B-Instruct   30.0    NaN 53.333333 instance_results_layer_30_alpha_0.5.jsonl
Qwen2.5-3B-Instruct   10.0    NaN 53.333333 instance_results_layer_10_alpha_1.5.jsonl
Qwen2.5-3B-Instruct   16.0    NaN 53.333333          results_layer_16_alpha_1.0.jsonl
Qwen2.5-3B-Instruct    NaN    NaN 53.000000          results_baseline_alpha_0.0.jsonl
Qwen2.5-3B-Instruct   30.0    2.0 52.666667            results_layer_30_alpha_2.jsonl
Qwen2.5-3B-Instruct   30.0    NaN 52.666667 instance_results_layer_30_alpha_1.5.jsonl
Qwen2.5-3B-Instruct   16.0    NaN 52.666667 instance_results_layer_16_alpha_1.5.jsonl
Qwen2.5-3B-Instruct   30.0    NaN

In [2]:
root_path = "./static/Qwen2.5-3B-Instruct/FOLIO" 
all_results_df = analyze_sweeps(root_path)

if all_results_df is not None:
    # --- 修改部分：按 Accuracy 降序排列 ---
    # ascending=False 表示降序
    all_results_df = all_results_df.sort_values(by='Accuracy', ascending=False)
    # ------------------------------------
    print(f"### 找到共 {len(all_results_df)} 条实验记录 ###")
    
    # 使用 pd.option_context 确保打印所有行，不会因为太长被折叠 (Optional)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
        print(all_results_df.to_string(index=False)) # 不打印索引号，更整洁
        
    # 如果你想顺便存个 CSV 方便在 Excel 看：
    # all_results_df.to_csv("all_experiment_results.csv", index=False)
else:
    print("未找到有效的结果文件。")

### 找到共 58 条实验记录 ###
               Task  Layer  Alpha  Accuracy                                      File
Qwen2.5-3B-Instruct    NaN    NaN 65.196078   results_repeat_baseline_alpha_0.0.jsonl
Qwen2.5-3B-Instruct   34.0    NaN 62.254902 instance_results_layer_34_alpha_0.5.jsonl
Qwen2.5-3B-Instruct   34.0    NaN 61.764706          results_layer_34_alpha_0.5.jsonl
Qwen2.5-3B-Instruct   26.0    NaN 61.764706          results_layer_26_alpha_1.5.jsonl
Qwen2.5-3B-Instruct   34.0    NaN 61.274510          results_layer_34_alpha_1.5.jsonl
Qwen2.5-3B-Instruct   20.0    NaN 61.274510          results_layer_20_alpha_1.0.jsonl
Qwen2.5-3B-Instruct    6.0    NaN 60.784314           results_layer_6_alpha_0.5.jsonl
Qwen2.5-3B-Instruct   30.0    NaN 60.784314 instance_results_layer_30_alpha_1.5.jsonl
Qwen2.5-3B-Instruct   30.0    NaN 60.784314          results_layer_30_alpha_0.5.jsonl
Qwen2.5-3B-Instruct   12.0    NaN 60.294118          results_layer_12_alpha_1.0.jsonl
Qwen2.5-3B-Instruct    NaN    NaN

In [9]:
root_path = "./static/Qwen2.5-3B-Instruct/AR-LSAT" 
all_results_df = analyze_sweeps(root_path)

if all_results_df is not None:
    # --- 修改部分：按 Accuracy 降序排列 ---
    # ascending=False 表示降序
    all_results_df = all_results_df.sort_values(by='Accuracy', ascending=False)
    # ------------------------------------
    print(f"### 找到共 {len(all_results_df)} 条实验记录 ###")
    
    # 使用 pd.option_context 确保打印所有行，不会因为太长被折叠 (Optional)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
        print(all_results_df.to_string(index=False)) # 不打印索引号，更整洁
        
    # 如果你想顺便存个 CSV 方便在 Excel 看：
    # all_results_df.to_csv("all_experiment_results.csv", index=False)
else:
    print("未找到有效的结果文件。")

### 找到共 60 条实验记录 ###
               Task  Layer  Alpha  Accuracy                                      File
Qwen2.5-3B-Instruct   26.0    NaN 27.826087          results_layer_26_alpha_1.0.jsonl
Qwen2.5-3B-Instruct    NaN    NaN 26.956522       results_64_baseline_alpha_0.0.jsonl
Qwen2.5-3B-Instruct   26.0    NaN 26.521739          results_layer_26_alpha_1.5.jsonl
Qwen2.5-3B-Instruct   12.0    NaN 26.086957          results_layer_12_alpha_0.5.jsonl
Qwen2.5-3B-Instruct    6.0    NaN 25.652174  instance_results_layer_6_alpha_1.5.jsonl
Qwen2.5-3B-Instruct    6.0    NaN 25.217391           results_layer_6_alpha_1.5.jsonl
Qwen2.5-3B-Instruct   30.0    NaN 25.217391 instance_results_layer_30_alpha_1.0.jsonl
Qwen2.5-3B-Instruct   24.0    NaN 25.217391 instance_results_layer_24_alpha_1.5.jsonl
Qwen2.5-3B-Instruct   30.0    NaN 24.347826          results_layer_30_alpha_0.5.jsonl
Qwen2.5-3B-Instruct   30.0    NaN 24.347826 instance_results_layer_30_alpha_0.5.jsonl
Qwen2.5-3B-Instruct   12.0    NaN

In [10]:
root_path = "./static/Qwen2.5-3B-Instruct/ProofWriter" 
all_results_df = analyze_sweeps(root_path)

if all_results_df is not None:
    # --- 修改部分：按 Accuracy 降序排列 ---
    # ascending=False 表示降序
    all_results_df = all_results_df.sort_values(by='Accuracy', ascending=False)
    # ------------------------------------
    print(f"### 找到共 {len(all_results_df)} 条实验记录 ###")
    
    # 使用 pd.option_context 确保打印所有行，不会因为太长被折叠 (Optional)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
        print(all_results_df.to_string(index=False)) # 不打印索引号，更整洁
        
    # 如果你想顺便存个 CSV 方便在 Excel 看：
    # all_results_df.to_csv("all_experiment_results.csv", index=False)
else:
    print("未找到有效的结果文件。")

### 找到共 30 条实验记录 ###
               Task  Layer  Alpha  Accuracy                                      File
Qwen2.5-3B-Instruct   20.0    NaN 59.500000 instance_results_layer_20_alpha_1.0.jsonl
Qwen2.5-3B-Instruct    NaN    NaN 57.666667   results_repeat_baseline_alpha_0.0.jsonl
Qwen2.5-3B-Instruct   30.0    NaN 57.500000 instance_results_layer_30_alpha_0.5.jsonl
Qwen2.5-3B-Instruct   30.0    NaN 57.166667 instance_results_layer_30_alpha_1.0.jsonl
Qwen2.5-3B-Instruct   30.0    NaN 56.333333 instance_results_layer_30_alpha_1.5.jsonl
Qwen2.5-3B-Instruct    NaN    NaN 56.000000        results_64baseline_alpha_0.0.jsonl
Qwen2.5-3B-Instruct   26.0    NaN 56.000000 instance_results_layer_26_alpha_0.5.jsonl
Qwen2.5-3B-Instruct   24.0    NaN 56.000000 instance_results_layer_24_alpha_1.0.jsonl
Qwen2.5-3B-Instruct   12.0    NaN 55.500000 instance_results_layer_12_alpha_1.5.jsonl
Qwen2.5-3B-Instruct   12.0    NaN 55.333333 instance_results_layer_12_alpha_0.5.jsonl
Qwen2.5-3B-Instruct   34.0    NaN

In [5]:
# 执行分析
root_path = "./static/Qwen2.5-7B-Instruct/LogicalDeduction" 
all_results_df = analyze_sweeps(root_path)

if all_results_df is not None:
    # --- 修改部分：按 Accuracy 降序排列 ---
    # ascending=False 表示降序
    all_results_df = all_results_df.sort_values(by='Accuracy', ascending=False)
    # ------------------------------------
    print(f"### 找到共 {len(all_results_df)} 条实验记录 ###")
    
    # 使用 pd.option_context 确保打印所有行，不会因为太长被折叠 (Optional)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
        print(all_results_df.to_string(index=False)) # 不打印索引号，更整洁
        
    # 如果你想顺便存个 CSV 方便在 Excel 看：
    # all_results_df.to_csv("all_experiment_results.csv", index=False)
else:
    print("未找到有效的结果文件。")

### 找到共 18 条实验记录 ###
               Task  Layer  Alpha  Accuracy                                     File
Qwen2.5-7B-Instruct    NaN    NaN 70.666667         results_baseline_alpha_0.0.jsonl
Qwen2.5-7B-Instruct   26.0    NaN 69.333333         results_layer_26_alpha_0.5.jsonl
Qwen2.5-7B-Instruct   20.0    NaN 69.333333         results_layer_20_alpha_1.0.jsonl
Qwen2.5-7B-Instruct   20.0    NaN 68.333333         results_layer_20_alpha_1.5.jsonl
Qwen2.5-7B-Instruct   26.0    NaN 68.000000         results_layer_26_alpha_1.5.jsonl
Qwen2.5-7B-Instruct   26.0    NaN 67.666667         results_layer_26_alpha_1.0.jsonl
Qwen2.5-7B-Instruct    6.0    NaN  0.000000          results_layer_6_alpha_0.5.jsonl
Qwen2.5-7B-Instruct    6.0    NaN  0.000000          results_layer_6_alpha_1.0.jsonl
Qwen2.5-7B-Instruct    NaN    NaN  0.000000 results_reverse_baseline_alpha_0.0.jsonl
Qwen2.5-7B-Instruct   20.0    NaN  0.000000         results_layer_20_alpha_0.5.jsonl
Qwen2.5-7B-Instruct   16.0    NaN  0.000000 

In [ ]:
# 执行分析
root_path = "./static/Qwen2.5-7B-Instruct/FOLIO" 
all_results_df = analyze_sweeps(root_path)

if all_results_df is not None:
    # --- 修改部分：按 Accuracy 降序排列 ---
    # ascending=False 表示降序
    all_results_df = all_results_df.sort_values(by='Accuracy', ascending=False)
    # ------------------------------------
    print(f"### 找到共 {len(all_results_df)} 条实验记录 ###")
    
    # 使用 pd.option_context 确保打印所有行，不会因为太长被折叠 (Optional)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
        print(all_results_df.to_string(index=False)) # 不打印索引号，更整洁
        
    # 如果你想顺便存个 CSV 方便在 Excel 看：
    # all_results_df.to_csv("all_experiment_results.csv", index=False)
else:
    print("未找到有效的结果文件。")

### 找到共 18 条实验记录 ###
               Task  Layer  Alpha  Accuracy                                     File
Qwen2.5-7B-Instruct    6.0    NaN 67.156863          results_layer_6_alpha_0.5.jsonl
Qwen2.5-7B-Instruct    6.0    NaN 67.647059          results_layer_6_alpha_1.0.jsonl
Qwen2.5-7B-Instruct    6.0    NaN 66.666667          results_layer_6_alpha_1.5.jsonl
Qwen2.5-7B-Instruct   10.0    NaN 65.196078         results_layer_10_alpha_0.5.jsonl
Qwen2.5-7B-Instruct   10.0    NaN 70.098039         results_layer_10_alpha_1.0.jsonl
Qwen2.5-7B-Instruct   10.0    NaN 69.117647         results_layer_10_alpha_1.5.jsonl
Qwen2.5-7B-Instruct   16.0    NaN 68.627451         results_layer_16_alpha_0.5.jsonl
Qwen2.5-7B-Instruct   16.0    NaN 69.607843         results_layer_16_alpha_1.0.jsonl
Qwen2.5-7B-Instruct   16.0    NaN 66.666667         results_layer_16_alpha_1.5.jsonl
Qwen2.5-7B-Instruct   20.0    NaN 66.666667         results_layer_20_alpha_0.5.jsonl
Qwen2.5-7B-Instruct   20.0    NaN 68.627451 